In [ ]:
import json
import os
import random
import time
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, RobertaModel,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Config ────────────────────────────────────────────────────────
DATASET        = "SST-2"
MODEL_CE       = "RoBERTa-CE"
MODEL_SCL      = "RoBERTa-SCL"
MODEL_NAME     = "roberta-base"
NUM_EPOCHS     = 3
BATCH_SIZE     = 32
LEARNING_RATE  = 2e-5
MAX_LENGTH     = 128
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.06
OPTIMIZER_NAME = "AdamW"
SCHEDULER_NAME = "linear"
TEST_LABELS    = False

# SCL-specific
TEMPERATURE   = 0.3
LAMBDA_WEIGHT = 0.5
PROJ_DIM      = 128

TEXT_COLUMN  = "sentence_original"
LABEL_COLUMN = "label"
ROOT_DIR     = os.getcwd()

print(f"Dataset:       {DATASET}")
print(f"Model:         {MODEL_NAME}")
print(f"Epochs:        {NUM_EPOCHS}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max length:    {MAX_LENGTH}")
print(f"Weight decay:  {WEIGHT_DECAY}")
print(f"Warmup ratio:  {WARMUP_RATIO}")
print(f"Temperature τ: {TEMPERATURE}")
print(f"Lambda λ:      {LAMBDA_WEIGHT}")
print(f"Proj dim:      {PROJ_DIM}")
print(f"Seed:          {SEED}")
print(f"Root dir:      {ROOT_DIR}")

In [ ]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    file_path = os.path.join(dataset_path, f"{split_name}.json")
    if not os.path.exists(file_path):
        print(f"  {split_name}.json not found in {dataset_path}")
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  {split_name}.json: {len(df)} samples")
    return df

print(f"Loading data from: {dataset_path}\n")
df_train      = load_split("train")
df_validation = load_split("validation")
df_test       = load_split("test")

if df_train is not None:
    print(f"\nFirst 3 train examples:")
    display(df_train.head(3))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SSTDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_texts  = df_train[TEXT_COLUMN].fillna("").tolist()
train_labels = df_train[LABEL_COLUMN].tolist()
val_texts    = df_validation[TEXT_COLUMN].fillna("").tolist()
val_labels   = df_validation[LABEL_COLUMN].tolist()
test_texts   = df_test[TEXT_COLUMN].fillna("").tolist()

y_test = None
if TEST_LABELS and LABEL_COLUMN in df_test.columns:
    y_test = df_test[LABEL_COLUMN].tolist()

train_dataset = SSTDataset(train_texts, train_labels)
val_dataset   = SSTDataset(val_texts, val_labels)
test_dataset  = SSTDataset(test_texts, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Tokenizer:     {MODEL_NAME}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# ── Supervised Contrastive Loss ───────────────────────────────────
# Gunel et al., "Supervised Contrastive Learning for Pre-trained Language Model Fine-tuning" (ICLR 2021)

class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.3):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        """
        embeddings: L2-normalized projected embeddings, shape (B, proj_dim)
        labels:     class labels, shape (B,)
        """
        sim_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature  # (B, B)

        labels_row = labels.unsqueeze(0)                              # (1, B)
        mask = torch.eq(labels_row, labels_row.T).float()             # (B, B)

        batch_size = embeddings.shape[0]
        self_mask  = torch.eye(batch_size, device=embeddings.device).bool()
        mask       = mask.masked_fill(self_mask, 0)
        sim_matrix = sim_matrix.masked_fill(self_mask, float('-inf'))

        # Numerical stability
        sim_max, _ = sim_matrix.max(dim=1, keepdim=True)
        sim_matrix = sim_matrix - sim_max.detach()

        exp_sim  = torch.exp(sim_matrix)
        exp_sim  = exp_sim.masked_fill(self_mask, 0)
        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

        positive_count = mask.sum(dim=1)
        mean_log_prob  = (mask * log_prob).sum(dim=1) / (positive_count + 1e-8)

        valid = positive_count > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        return -mean_log_prob[valid].mean()

In [ ]:
# ── RoBERTa model with projection head for SCL ────────────────────

class RoBERTaForSCL(nn.Module):
    def __init__(self, model_name, num_labels, proj_dim=128):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained(model_name)
        hidden_size     = self.roberta.config.hidden_size

        # Classification head on raw [CLS] embedding
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        )

        # Projection head (training only): hidden → proj_dim → proj_dim with ReLU
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_size, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, input_ids, attention_mask, return_projected=False):
        outputs       = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        logits        = self.classifier(cls_embedding)

        if return_projected:
            projected = self.projection_head(cls_embedding)
            projected = F.normalize(projected, p=2, dim=1)
            return logits, projected, cls_embedding

        return logits, cls_embedding

In [ ]:
# ── Optimizer / Scheduler builders + save_results ─────────────────

def build_optimizer(name, model_params, lr, weight_decay):
    if name == "AdamW":
        return torch.optim.AdamW(model_params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

def build_scheduler(name, optimizer, warmup_steps, total_steps):
    if name == "linear":
        return get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    raise ValueError(f"Unknown scheduler: {name}")


def save_results(model_obj, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, model_hf_name, num_epochs, batch_size, learning_rate, max_length,
                 test_labels, text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir,
                 optimizer_info=None, scheduler_info=None,
                 extra_params=None):

    model_size_bytes = sum(p.numel() * p.element_size() for p in model_obj.parameters())
    model_size_mb    = model_size_bytes / (1024 * 1024)

    timestamp_str = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    run_name      = f"{model_hf_name}_ep{num_epochs}_bs{batch_size}_lr{learning_rate}_{timestamp_str}"

    output_dir = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(output_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(output_dir, fig_name), dpi=150, bbox_inches="tight")
    print(f"Charts saved: {list(figures.keys())}")

    total_params     = sum(p.numel() for p in model_obj.parameters())
    trainable_params = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

    base_params = {
        "num_epochs":       num_epochs,
        "batch_size":       batch_size,
        "learning_rate":    learning_rate,
        "max_length":       max_length,
        "total_params":     total_params,
        "trainable_params": trainable_params,
    }
    if extra_params:
        base_params.update(extra_params)

    metrics = {
        "model":          model_name,
        "model_hf_name":  model_hf_name,
        "dataset":        dataset,
        "run_name":       run_name,
        "timestamp":      timestamp_str,
        "seed":           SEED,
        "model_params":   base_params,
        "model_size":     {"bytes": model_size_bytes, "MB": round(model_size_mb, 4)},
        "training": {
            "training_time_seconds": round(train_time, 4),
            "entries_per_second":    round(entries_per_second, 2),
            "num_epochs":            num_epochs,
            "train_samples":         len(df_train),
        },
        "data": {
            "train_samples":      len(df_train),
            "validation_samples": len(df_validation),
            "test_samples":       len(df_test) if df_test is not None else 0,
            "text_column":        text_column,
            "label_column":       label_column,
            "classes":            class_labels,
        },
        "optimizer":          optimizer_info or {},
        "scheduler":          scheduler_info or {},
        "train_results":      train_results,
        "validation_results": val_results,
    }

    if test_labels and test_results is not None:
        metrics["test_results"] = test_results

    metrics_path = os.path.join(output_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"metrics.json saved in: {output_dir}")
    return output_dir, metrics_path, run_name

In [ ]:
# ── PART 1: Train CE Baseline ─────────────────────────────────────
print(f"\n{'='*60}")
print(f"  TRAINING CE BASELINE (RoBERTa-CE)")
print(f"{'='*60}")

set_seed(SEED)

ce_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
ce_model.to(device)

total_steps_ce  = len(train_loader) * NUM_EPOCHS
warmup_steps_ce = int(WARMUP_RATIO * total_steps_ce)

ce_optimizer = build_optimizer(OPTIMIZER_NAME, ce_model.parameters(), LEARNING_RATE, WEIGHT_DECAY)
ce_scheduler = build_scheduler(SCHEDULER_NAME, ce_optimizer, warmup_steps_ce, total_steps_ce)

ce_train_losses, ce_val_losses = [], []
ce_train_accs,   ce_val_accs   = [], []

start_time_ce = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    ce_model.train()
    epoch_loss = 0
    epoch_preds, epoch_labels_list = [], []

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        ce_optimizer.zero_grad()
        outputs = ce_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(ce_model.parameters(), 1.0)
        ce_optimizer.step()
        ce_scheduler.step()

        epoch_loss += outputs.loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        epoch_preds.extend(preds.cpu().numpy())
        epoch_labels_list.extend(labels.cpu().numpy())

    train_loss = epoch_loss / len(train_loader)
    train_acc  = accuracy_score(epoch_labels_list, epoch_preds)
    ce_train_losses.append(train_loss)
    ce_train_accs.append(train_acc)

    ce_model.eval()
    val_loss_sum = 0
    val_preds, val_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            outputs        = ce_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss_sum  += outputs.loss.item()
            val_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_loss = val_loss_sum / len(val_loader)
    val_acc  = accuracy_score(val_labels_list, val_preds)
    ce_val_losses.append(val_loss)
    ce_val_accs.append(val_acc)

    print(f"  Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

train_time_ce          = time.time() - start_time_ce
entries_per_second_ce  = (len(train_labels) * NUM_EPOCHS) / train_time_ce
print(f"\nDone in {train_time_ce:.2f}s  ({entries_per_second_ce:.0f} entries/sec)")

In [ ]:
# ── Evaluate CE Baseline ──────────────────────────────────────────
ce_model.eval()

def predict_ce(model, loader):
    all_preds, all_labels_out = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            if "labels" in batch:
                labels  = batch["labels"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_loss += outputs.loss.item()
                all_labels_out.extend(labels.cpu().numpy())
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
    avg_loss = total_loss / len(loader) if all_labels_out else None
    return np.array(all_preds), np.array(all_labels_out) if all_labels_out else None, avg_loss

y_train_pred_ce, y_train_true_ce, train_loss_final_ce = predict_ce(ce_model, train_loader)
y_val_pred_ce,   y_val_true_ce,   val_loss_final_ce   = predict_ce(ce_model, val_loader)
y_test_pred_ce,  _,               _                   = predict_ce(ce_model, test_loader)

ce_train_accuracy      = accuracy_score(y_train_true_ce, y_train_pred_ce)
ce_val_accuracy        = accuracy_score(y_val_true_ce,   y_val_pred_ce)
ce_val_precision_macro = precision_score(y_val_true_ce,  y_val_pred_ce, average="macro", zero_division=0)
ce_val_recall_macro    = recall_score(y_val_true_ce,     y_val_pred_ce, average="macro", zero_division=0)
ce_val_f1_macro        = f1_score(y_val_true_ce,         y_val_pred_ce, average="macro", zero_division=0)
ce_val_precision_pc    = precision_score(y_val_true_ce,  y_val_pred_ce, average=None, zero_division=0).tolist()
ce_val_recall_pc       = recall_score(y_val_true_ce,     y_val_pred_ce, average=None, zero_division=0).tolist()
ce_val_f1_pc           = f1_score(y_val_true_ce,         y_val_pred_ce, average=None, zero_division=0).tolist()
ce_val_conf_matrix     = confusion_matrix(y_val_true_ce, y_val_pred_ce).tolist()
class_labels           = sorted(np.unique(y_val_true_ce).tolist())

print("=" * 60)
print(f"  RESULTS RoBERTa-CE on {DATASET}")
print("=" * 60)
print(f"  Train Accuracy: {ce_train_accuracy:.4f}  | Loss: {train_loss_final_ce:.4f}")
print(f"  Val Accuracy:   {ce_val_accuracy:.4f}  | Loss: {val_loss_final_ce:.4f}")
print(f"  Val F1 (macro): {ce_val_f1_macro:.4f}")
print(f"\n  Classification Report (VAL):")
print(classification_report(y_val_true_ce, y_val_pred_ce, zero_division=0))

In [ ]:
# ── Plots & save CE Baseline ──────────────────────────────────────
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig_cm_ce, ax_cm_ce = plt.subplots(figsize=(6, 5))
sns.heatmap(np.array(ce_val_conf_matrix), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels, ax=ax_cm_ce)
ax_cm_ce.set_xlabel("Predicted")
ax_cm_ce.set_ylabel("Actual")
ax_cm_ce.set_title(f"{MODEL_CE} - {DATASET}", fontweight="bold", color="black")
fig_cm_ce.tight_layout()
plt.show()

fig_acc_ce, ax_acc_ce = plt.subplots(figsize=(9, 5))
ax_acc_ce.plot(epochs_range, ce_train_accs, label="Train Accuracy", linewidth=2, color="#1f77b4")
ax_acc_ce.plot(epochs_range, ce_val_accs,   label="Validation Accuracy", linewidth=2, color="#ff7f0e")
ax_acc_ce.set_xlabel("Epoch")
ax_acc_ce.set_ylabel("Accuracy")
ax_acc_ce.set_title(f"{MODEL_CE} - {DATASET}", fontweight="bold", color="black")
ax_acc_ce.legend(loc="lower right")
ax_acc_ce.grid(True, alpha=0.3)
fig_acc_ce.tight_layout()
plt.show()

fig_loss_ce, ax_loss_ce = plt.subplots(figsize=(9, 5))
ax_loss_ce.plot(epochs_range, ce_train_losses, label="Train Loss", linewidth=2, color="#1f77b4")
ax_loss_ce.plot(epochs_range, ce_val_losses,   label="Validation Loss", linewidth=2, color="#ff7f0e")
ax_loss_ce.set_xlabel("Epoch")
ax_loss_ce.set_ylabel("Loss (CrossEntropy)")
ax_loss_ce.set_title(f"{MODEL_CE} - {DATASET}", fontweight="bold", color="black")
ax_loss_ce.legend(loc="upper right")
ax_loss_ce.grid(True, alpha=0.3)
fig_loss_ce.tight_layout()
plt.show()

figures_ce   = {"confusion_matrix.png": fig_cm_ce, "accuracy_curves.png": fig_acc_ce, "loss_curves.png": fig_loss_ce}
train_res_ce = {"accuracy": round(ce_train_accuracy, 4), "loss": round(train_loss_final_ce, 4)}
val_res_ce   = {
    "accuracy":            round(ce_val_accuracy, 4),
    "loss":                round(val_loss_final_ce, 4),
    "precision_macro":     round(ce_val_precision_macro, 4),
    "recall_macro":        round(ce_val_recall_macro, 4),
    "f1_macro":            round(ce_val_f1_macro, 4),
    "precision_per_class": [round(p, 4) for p in ce_val_precision_pc],
    "recall_per_class":    [round(r, 4) for r in ce_val_recall_pc],
    "f1_per_class":        [round(f, 4) for f in ce_val_f1_pc],
    "confusion_matrix":    ce_val_conf_matrix,
}
opt_info_ce   = {"name": OPTIMIZER_NAME, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}
sched_info_ce = {"name": SCHEDULER_NAME, "warmup_steps": warmup_steps_ce, "total_steps": total_steps_ce}

output_dir_ce, metrics_path_ce, run_name_ce = save_results(
    model_obj=ce_model, figures=figures_ce,
    train_results=train_res_ce, val_results=val_res_ce, test_results=None,
    train_time=train_time_ce, entries_per_second=entries_per_second_ce,
    dataset=DATASET, model_name=MODEL_CE, model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,
    optimizer_info=opt_info_ce, scheduler_info=sched_info_ce,
)

In [ ]:
# ── PART 2: Train SCL Model ───────────────────────────────────────
print(f"\n{'='*60}")
print(f"  TRAINING SCL MODEL (RoBERTa-SCL)")
print(f"{'='*60}")

set_seed(SEED)

scl_model = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM)
scl_model.to(device)

total_steps_scl  = len(train_loader) * NUM_EPOCHS
warmup_steps_scl = int(WARMUP_RATIO * total_steps_scl)

scl_optimizer = build_optimizer(OPTIMIZER_NAME, scl_model.parameters(), LEARNING_RATE, WEIGHT_DECAY)
scl_scheduler = build_scheduler(SCHEDULER_NAME, scl_optimizer, warmup_steps_scl, total_steps_scl)

scl_criterion = SupervisedContrastiveLoss(temperature=TEMPERATURE)
ce_criterion  = nn.CrossEntropyLoss()

scl_train_losses,     scl_val_losses     = [], []
scl_train_accs,       scl_val_accs       = [], []
scl_ce_loss_history,  scl_scl_loss_history = [], []  # per-epoch breakdown

start_time_scl = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    scl_model.train()
    epoch_ce_loss, epoch_scl_loss = 0, 0
    epoch_preds, epoch_labels_list = [], []

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        scl_optimizer.zero_grad()
        logits, projected, _ = scl_model(input_ids, attention_mask, return_projected=True)

        loss_ce  = ce_criterion(logits, labels)
        loss_scl = scl_criterion(projected, labels)
        loss     = loss_ce + LAMBDA_WEIGHT * loss_scl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(scl_model.parameters(), 1.0)
        scl_optimizer.step()
        scl_scheduler.step()

        epoch_ce_loss  += loss_ce.item()
        epoch_scl_loss += loss_scl.item()
        epoch_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        epoch_labels_list.extend(labels.cpu().numpy())

    avg_ce_loss  = epoch_ce_loss  / len(train_loader)
    avg_scl_loss = epoch_scl_loss / len(train_loader)
    train_acc    = accuracy_score(epoch_labels_list, epoch_preds)

    scl_train_losses.append(avg_ce_loss + LAMBDA_WEIGHT * avg_scl_loss)
    scl_train_accs.append(train_acc)
    scl_ce_loss_history.append(avg_ce_loss)
    scl_scl_loss_history.append(avg_scl_loss)

    scl_model.eval()
    val_loss_sum = 0
    val_preds, val_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            logits, _      = scl_model(input_ids, attention_mask, return_projected=False)
            val_loss_sum  += ce_criterion(logits, labels).item()
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_loss = val_loss_sum / len(val_loader)
    val_acc  = accuracy_score(val_labels_list, val_preds)
    scl_val_losses.append(val_loss)
    scl_val_accs.append(val_acc)

    print(f"  Epoch {epoch}/{NUM_EPOCHS} | "
          f"CE: {avg_ce_loss:.4f} | SCL: {avg_scl_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

train_time_scl         = time.time() - start_time_scl
entries_per_second_scl = (len(train_labels) * NUM_EPOCHS) / train_time_scl
print(f"\nDone in {train_time_scl:.2f}s  ({entries_per_second_scl:.0f} entries/sec)")

In [ ]:
# ── Evaluate SCL Model ────────────────────────────────────────────
scl_model.eval()

def predict_scl(model, loader):
    all_preds, all_labels_out = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits, _      = model(input_ids, attention_mask, return_projected=False)
            if "labels" in batch:
                labels = batch["labels"].to(device)
                total_loss += ce_criterion(logits, labels).item()
                all_labels_out.extend(labels.cpu().numpy())
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
    avg_loss = total_loss / len(loader) if all_labels_out else None
    return np.array(all_preds), np.array(all_labels_out) if all_labels_out else None, avg_loss

y_train_pred_scl, y_train_true_scl, train_loss_final_scl = predict_scl(scl_model, train_loader)
y_val_pred_scl,   y_val_true_scl,   val_loss_final_scl   = predict_scl(scl_model, val_loader)
y_test_pred_scl,  _,                _                    = predict_scl(scl_model, test_loader)

scl_train_accuracy      = accuracy_score(y_train_true_scl, y_train_pred_scl)
scl_val_accuracy        = accuracy_score(y_val_true_scl,   y_val_pred_scl)
scl_val_precision_macro = precision_score(y_val_true_scl,  y_val_pred_scl, average="macro", zero_division=0)
scl_val_recall_macro    = recall_score(y_val_true_scl,     y_val_pred_scl, average="macro", zero_division=0)
scl_val_f1_macro        = f1_score(y_val_true_scl,         y_val_pred_scl, average="macro", zero_division=0)
scl_val_precision_pc    = precision_score(y_val_true_scl,  y_val_pred_scl, average=None, zero_division=0).tolist()
scl_val_recall_pc       = recall_score(y_val_true_scl,     y_val_pred_scl, average=None, zero_division=0).tolist()
scl_val_f1_pc           = f1_score(y_val_true_scl,         y_val_pred_scl, average=None, zero_division=0).tolist()
scl_val_conf_matrix     = confusion_matrix(y_val_true_scl, y_val_pred_scl).tolist()

print("=" * 60)
print(f"  RESULTS RoBERTa-SCL on {DATASET}")
print("=" * 60)
print(f"  Train Accuracy: {scl_train_accuracy:.4f}  | Loss: {train_loss_final_scl:.4f}")
print(f"  Val Accuracy:   {scl_val_accuracy:.4f}  | Loss: {val_loss_final_scl:.4f}")
print(f"  Val F1 (macro): {scl_val_f1_macro:.4f}")
print(f"\n  Classification Report (VAL):")
print(classification_report(y_val_true_scl, y_val_pred_scl, zero_division=0))

print(f"\n  CE vs SCL Comparison:")
print(f"  {'Metric':<20} {'RoBERTa-CE':>12} {'RoBERTa-SCL':>12} {'Δ':>10}")
print(f"  {'-'*56}")
for label, ce_v, scl_v in [
    ("Val Accuracy",     ce_val_accuracy,        scl_val_accuracy),
    ("Val F1 (macro)",   ce_val_f1_macro,         scl_val_f1_macro),
    ("Val Precision",    ce_val_precision_macro,  scl_val_precision_macro),
    ("Val Recall",       ce_val_recall_macro,     scl_val_recall_macro),
]:
    print(f"  {label:<20} {ce_v:>12.4f} {scl_v:>12.4f} {scl_v - ce_v:>+10.4f}")

In [ ]:
# ── Plots & save SCL ──────────────────────────────────────────────
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig_cm_scl, ax_cm_scl = plt.subplots(figsize=(6, 5))
sns.heatmap(np.array(scl_val_conf_matrix), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels, ax=ax_cm_scl)
ax_cm_scl.set_xlabel("Predicted")
ax_cm_scl.set_ylabel("Actual")
ax_cm_scl.set_title(f"{MODEL_SCL} - {DATASET}", fontweight="bold", color="black")
fig_cm_scl.tight_layout()
plt.show()

fig_acc_scl, ax_acc_scl = plt.subplots(figsize=(9, 5))
ax_acc_scl.plot(epochs_range, scl_train_accs, label="Train Accuracy", linewidth=2, color="#1f77b4")
ax_acc_scl.plot(epochs_range, scl_val_accs,   label="Validation Accuracy", linewidth=2, color="#ff7f0e")
ax_acc_scl.set_xlabel("Epoch")
ax_acc_scl.set_ylabel("Accuracy")
ax_acc_scl.set_title(f"{MODEL_SCL} - {DATASET}", fontweight="bold", color="black")
ax_acc_scl.legend(loc="lower right")
ax_acc_scl.grid(True, alpha=0.3)
fig_acc_scl.tight_layout()
plt.show()

fig_loss_scl, ax_loss_scl = plt.subplots(figsize=(9, 5))
ax_loss_scl.plot(epochs_range, scl_train_losses, label="Train Loss (CE + λ·SCL)", linewidth=2, color="#1f77b4")
ax_loss_scl.plot(epochs_range, scl_val_losses,   label="Validation Loss (CE)",    linewidth=2, color="#ff7f0e")
ax_loss_scl.set_xlabel("Epoch")
ax_loss_scl.set_ylabel("Loss")
ax_loss_scl.set_title(f"{MODEL_SCL} - {DATASET}", fontweight="bold", color="black")
ax_loss_scl.legend(loc="upper right")
ax_loss_scl.grid(True, alpha=0.3)
fig_loss_scl.tight_layout()
plt.show()

fig_scl_breakdown, ax_scl_bd = plt.subplots(figsize=(9, 5))
ax_scl_bd.plot(epochs_range, scl_ce_loss_history,  label="CE Loss",  linewidth=2, color="#2ca02c")
ax_scl_bd.plot(epochs_range, scl_scl_loss_history, label="SCL Loss", linewidth=2, color="#d62728")
ax_scl_bd.set_xlabel("Epoch")
ax_scl_bd.set_ylabel("Loss")
ax_scl_bd.set_title(f"{MODEL_SCL} - Loss Breakdown", fontweight="bold", color="black")
ax_scl_bd.legend(loc="upper right")
ax_scl_bd.grid(True, alpha=0.3)
fig_scl_breakdown.tight_layout()
plt.show()

figures_scl   = {
    "confusion_matrix.png":   fig_cm_scl,
    "accuracy_curves.png":    fig_acc_scl,
    "loss_curves.png":        fig_loss_scl,
    "scl_loss_breakdown.png": fig_scl_breakdown,
}
train_res_scl = {"accuracy": round(scl_train_accuracy, 4), "loss": round(train_loss_final_scl, 4)}
val_res_scl   = {
    "accuracy":            round(scl_val_accuracy, 4),
    "loss":                round(val_loss_final_scl, 4),
    "precision_macro":     round(scl_val_precision_macro, 4),
    "recall_macro":        round(scl_val_recall_macro, 4),
    "f1_macro":            round(scl_val_f1_macro, 4),
    "precision_per_class": [round(p, 4) for p in scl_val_precision_pc],
    "recall_per_class":    [round(r, 4) for r in scl_val_recall_pc],
    "f1_per_class":        [round(f, 4) for f in scl_val_f1_pc],
    "confusion_matrix":    scl_val_conf_matrix,
}
opt_info_scl   = {"name": OPTIMIZER_NAME, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}
sched_info_scl = {"name": SCHEDULER_NAME, "warmup_steps": warmup_steps_scl, "total_steps": total_steps_scl}
extra_scl      = {
    "temperature":  TEMPERATURE,
    "lambda_weight": LAMBDA_WEIGHT,
    "proj_dim":     PROJ_DIM,
    "loss":         f"CE + {LAMBDA_WEIGHT}*SCL",
}

output_dir_scl, metrics_path_scl, run_name_scl = save_results(
    model_obj=scl_model, figures=figures_scl,
    train_results=train_res_scl, val_results=val_res_scl, test_results=None,
    train_time=train_time_scl, entries_per_second=entries_per_second_scl,
    dataset=DATASET, model_name=MODEL_SCL, model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,
    optimizer_info=opt_info_scl, scheduler_info=sched_info_scl,
    extra_params=extra_scl,
)

In [ ]:
# ── Experiment 2: Few-shot Learning ──────────────────────────────
print("\n" + "="*60)
print("  EXPERIMENT 2: Few-shot Learning")
print("="*60)

FRACTIONS       = [0.01, 0.05, 0.10, 0.50, 1.0]
FEW_SHOT_SEEDS  = [42, 123, 456]
FEW_SHOT_EPOCHS = 3

def get_stratified_subset(texts, labels, fraction, seed):
    if fraction >= 1.0:
        return texts, labels
    _, sub_texts, _, sub_labels = train_test_split(
        texts, labels, test_size=fraction, stratify=labels, random_state=seed
    )
    return sub_texts, sub_labels

def train_eval_ce(sub_texts, sub_labels, n_epochs, seed):
    set_seed(seed)
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    m.to(device)
    ds     = SSTDataset(sub_texts, sub_labels)
    loader = DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=True)
    tot    = len(loader) * n_epochs
    opt    = build_optimizer(OPTIMIZER_NAME, m.parameters(), LEARNING_RATE, WEIGHT_DECAY)
    sch    = build_scheduler(SCHEDULER_NAME, opt, int(WARMUP_RATIO * tot), tot)
    m.train()
    for _ in range(n_epochs):
        for batch in loader:
            ids, mask, lbs = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            opt.zero_grad(); out = m(input_ids=ids, attention_mask=mask, labels=lbs)
            out.loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step(); sch.step()
    m.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = m(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            trues.extend(batch["labels"].numpy())
    return accuracy_score(trues, preds)

def train_eval_scl(sub_texts, sub_labels, n_epochs, seed):
    set_seed(seed)
    m = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM)
    m.to(device)
    ds     = SSTDataset(sub_texts, sub_labels)
    loader = DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=True)
    tot    = len(loader) * n_epochs
    opt    = build_optimizer(OPTIMIZER_NAME, m.parameters(), LEARNING_RATE, WEIGHT_DECAY)
    sch    = build_scheduler(SCHEDULER_NAME, opt, int(WARMUP_RATIO * tot), tot)
    scl_fn = SupervisedContrastiveLoss(temperature=TEMPERATURE)
    ce_fn  = nn.CrossEntropyLoss()
    m.train()
    for _ in range(n_epochs):
        for batch in loader:
            ids, mask, lbs = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            opt.zero_grad()
            logits, proj, _ = m(ids, mask, return_projected=True)
            (ce_fn(logits, lbs) + LAMBDA_WEIGHT * scl_fn(proj, lbs)).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step(); sch.step()
    m.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits, _ = m(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            trues.extend(batch["labels"].numpy())
    return accuracy_score(trues, preds)

fewshot_results = {f: {"CE": [], "SCL": []} for f in FRACTIONS}

for frac in FRACTIONS:
    print(f"\n  Fraction {frac*100:.0f}%:")
    for seed in FEW_SHOT_SEEDS:
        sub_t, sub_l = get_stratified_subset(train_texts, train_labels, frac, seed)
        print(f"    Seed {seed}: {len(sub_t)} samples", end="", flush=True)
        acc_ce  = train_eval_ce(sub_t,  sub_l, FEW_SHOT_EPOCHS, seed)
        acc_scl = train_eval_scl(sub_t, sub_l, FEW_SHOT_EPOCHS, seed)
        fewshot_results[frac]["CE"].append(acc_ce)
        fewshot_results[frac]["SCL"].append(acc_scl)
        print(f" | CE: {acc_ce:.4f} | SCL: {acc_scl:.4f}")

print("\n  Few-shot Summary:")
print(f"  {'Frac':>8}  {'CE Mean':>10} {'±':>4} {'SCL Mean':>10} {'±':>4}")
for frac in FRACTIONS:
    print(f"  {frac*100:>7.0f}%  "
          f"{np.mean(fewshot_results[frac]['CE']):>10.4f} "
          f"{np.std(fewshot_results[frac]['CE']):>6.4f}  "
          f"{np.mean(fewshot_results[frac]['SCL']):>10.4f} "
          f"{np.std(fewshot_results[frac]['SCL']):>6.4f}")

In [ ]:
# ── Few-shot plot + CSV ───────────────────────────────────────────
fracs_pct = [f * 100 for f in FRACTIONS]
ce_means  = [np.mean(fewshot_results[f]["CE"])  for f in FRACTIONS]
ce_stds   = [np.std(fewshot_results[f]["CE"])   for f in FRACTIONS]
scl_means = [np.mean(fewshot_results[f]["SCL"]) for f in FRACTIONS]
scl_stds  = [np.std(fewshot_results[f]["SCL"])  for f in FRACTIONS]

fig_fs, ax_fs = plt.subplots(figsize=(9, 5))
ax_fs.errorbar(fracs_pct, ce_means,  yerr=ce_stds,  label="RoBERTa-CE",  marker="o", linewidth=2, capsize=5, color="#1f77b4")
ax_fs.errorbar(fracs_pct, scl_means, yerr=scl_stds, label="RoBERTa-SCL", marker="s", linewidth=2, capsize=5, color="#d62728")
ax_fs.set_xlabel("Training data fraction (%)")
ax_fs.set_ylabel("Validation Accuracy")
ax_fs.set_title("Few-shot Learning: CE vs SCL", fontweight="bold", color="black")
ax_fs.legend()
ax_fs.grid(True, alpha=0.3)
fig_fs.tight_layout()
plt.show()

fig_fs.savefig(os.path.join(output_dir_scl, "fewshot_accuracy.png"), dpi=150, bbox_inches="tight")

fs_rows = []
for frac in FRACTIONS:
    for seed, ce_acc, scl_acc in zip(FEW_SHOT_SEEDS,
                                      fewshot_results[frac]["CE"],
                                      fewshot_results[frac]["SCL"]):
        fs_rows.append({"fraction": frac, "seed": seed, "CE_accuracy": ce_acc, "SCL_accuracy": scl_acc})

pd.DataFrame(fs_rows).to_csv(os.path.join(output_dir_scl, "fewshot_results.csv"), index=False)
print("Few-shot results saved.")

In [ ]:
# ── Experiment 3: Noise Robustness ───────────────────────────────
print("\n" + "="*60)
print("  EXPERIMENT 3: Noise Robustness")
print("="*60)

NOISE_LEVELS  = [0.0, 0.05, 0.10, 0.15]
NOISE_EPOCHS  = 3

def add_label_noise(labels, noise_rate, seed):
    rng   = np.random.RandomState(seed)
    lbs   = np.array(labels)
    n_flip = int(noise_rate * len(lbs))
    if n_flip > 0:
        flip_idx = rng.choice(len(lbs), n_flip, replace=False)
        lbs[flip_idx] = 1 - lbs[flip_idx]
    return lbs.tolist()

noise_results = {nl: {"CE": None, "SCL": None} for nl in NOISE_LEVELS}

for noise_rate in NOISE_LEVELS:
    print(f"\n  Noise {noise_rate*100:.0f}%:")
    noisy_labels = add_label_noise(train_labels, noise_rate, SEED)
    noisy_ds     = SSTDataset(train_texts, noisy_labels)
    noisy_loader = DataLoader(noisy_ds, batch_size=BATCH_SIZE, shuffle=True)
    tot_n   = len(noisy_loader) * NOISE_EPOCHS
    warm_n  = int(WARMUP_RATIO * tot_n)

    # CE on noisy data
    set_seed(SEED)
    m_ce = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    m_ce.to(device)
    opt_ce = build_optimizer(OPTIMIZER_NAME, m_ce.parameters(), LEARNING_RATE, WEIGHT_DECAY)
    sch_ce = build_scheduler(SCHEDULER_NAME, opt_ce, warm_n, tot_n)
    m_ce.train()
    for _ in range(NOISE_EPOCHS):
        for batch in noisy_loader:
            ids, mask, lbs = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            opt_ce.zero_grad(); out = m_ce(input_ids=ids, attention_mask=mask, labels=lbs)
            out.loss.backward(); torch.nn.utils.clip_grad_norm_(m_ce.parameters(), 1.0)
            opt_ce.step(); sch_ce.step()
    m_ce.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = m_ce(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            trues.extend(batch["labels"].numpy())
    noise_results[noise_rate]["CE"] = accuracy_score(trues, preds)
    print(f"    CE:  {noise_results[noise_rate]['CE']:.4f}")

    # SCL on noisy data
    set_seed(SEED)
    m_scl = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM)
    m_scl.to(device)
    opt_scl = build_optimizer(OPTIMIZER_NAME, m_scl.parameters(), LEARNING_RATE, WEIGHT_DECAY)
    sch_scl = build_scheduler(SCHEDULER_NAME, opt_scl, warm_n, tot_n)
    scl_fn_n = SupervisedContrastiveLoss(temperature=TEMPERATURE)
    ce_fn_n  = nn.CrossEntropyLoss()
    m_scl.train()
    for _ in range(NOISE_EPOCHS):
        for batch in noisy_loader:
            ids, mask, lbs = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            opt_scl.zero_grad()
            logits, proj, _ = m_scl(ids, mask, return_projected=True)
            (ce_fn_n(logits, lbs) + LAMBDA_WEIGHT * scl_fn_n(proj, lbs)).backward()
            torch.nn.utils.clip_grad_norm_(m_scl.parameters(), 1.0)
            opt_scl.step(); sch_scl.step()
    m_scl.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits, _ = m_scl(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            trues.extend(batch["labels"].numpy())
    noise_results[noise_rate]["SCL"] = accuracy_score(trues, preds)
    print(f"    SCL: {noise_results[noise_rate]['SCL']:.4f}")

print("\n  Noise Robustness Summary:")
print(f"  {'Noise %':>8}  {'CE Acc':>10} {'SCL Acc':>10} {'Δ':>10}")
for nl in NOISE_LEVELS:
    delta = noise_results[nl]['SCL'] - noise_results[nl]['CE']
    print(f"  {nl*100:>7.0f}%  {noise_results[nl]['CE']:>10.4f} {noise_results[nl]['SCL']:>10.4f} {delta:>+10.4f}")

In [ ]:
# ── Noise robustness plot + CSV ───────────────────────────────────
noise_pct      = [nl * 100 for nl in NOISE_LEVELS]
ce_noise_accs  = [noise_results[nl]["CE"]  for nl in NOISE_LEVELS]
scl_noise_accs = [noise_results[nl]["SCL"] for nl in NOISE_LEVELS]

fig_noise, ax_noise = plt.subplots(figsize=(9, 5))
ax_noise.plot(noise_pct, ce_noise_accs,  label="RoBERTa-CE",  marker="o", linewidth=2, color="#1f77b4")
ax_noise.plot(noise_pct, scl_noise_accs, label="RoBERTa-SCL", marker="s", linewidth=2, color="#d62728")
ax_noise.set_xlabel("Label Noise (%)")
ax_noise.set_ylabel("Validation Accuracy (clean labels)")
ax_noise.set_title("Noise Robustness: CE vs SCL", fontweight="bold", color="black")
ax_noise.legend()
ax_noise.grid(True, alpha=0.3)
fig_noise.tight_layout()
plt.show()

fig_noise.savefig(os.path.join(output_dir_scl, "noise_robustness.png"), dpi=150, bbox_inches="tight")

pd.DataFrame([
    {"noise_rate": nl, "CE_accuracy": noise_results[nl]["CE"], "SCL_accuracy": noise_results[nl]["SCL"]}
    for nl in NOISE_LEVELS
]).to_csv(os.path.join(output_dir_scl, "noise_robustness.csv"), index=False)
print("Noise robustness results saved.")

In [ ]:
# ── Experiment 4: t-SNE Visualization ────────────────────────────
print("\n" + "="*60)
print("  EXPERIMENT 4: t-SNE Visualization")
print("="*60)

# Extract [CLS] embeddings from CE model (AutoModelForSequenceClassification)
print("Extracting CE embeddings...")
ce_model.eval()
ce_embs, ce_emb_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        out  = ce_model.roberta(input_ids=ids, attention_mask=mask)
        ce_embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        ce_emb_labels.extend(batch["labels"].numpy())
ce_embs       = np.vstack(ce_embs)
ce_emb_labels = np.array(ce_emb_labels)

# Extract [CLS] embeddings from SCL model
print("Extracting SCL embeddings...")
scl_model.eval()
scl_embs, scl_emb_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        _, _, cls_emb = scl_model(ids, mask, return_projected=True)
        scl_embs.append(cls_emb.cpu().numpy())
        scl_emb_labels.extend(batch["labels"].numpy())
scl_embs       = np.vstack(scl_embs)
scl_emb_labels = np.array(scl_emb_labels)

print(f"CE embeddings:  {ce_embs.shape}")
print(f"SCL embeddings: {scl_embs.shape}")

print("Running t-SNE (this may take ~1 min)...")
ce_tsne  = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000).fit_transform(ce_embs)
scl_tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000).fit_transform(scl_embs)
print("t-SNE done.")

In [ ]:
# ── t-SNE plot ────────────────────────────────────────────────────
colors      = {0: "#d62728", 1: "#1f77b4"}   # red=negative, blue=positive
label_names = {0: "Negative", 1: "Positive"}

fig_tsne, axes_tsne = plt.subplots(1, 2, figsize=(14, 6))

for ax, emb, lbs, title in [
    (axes_tsne[0], ce_tsne,  ce_emb_labels,  "RoBERTa-CE (CLS embeddings)"),
    (axes_tsne[1], scl_tsne, scl_emb_labels, "RoBERTa-SCL (CLS embeddings)"),
]:
    for label in [0, 1]:
        mask_l = lbs == label
        ax.scatter(emb[mask_l, 0], emb[mask_l, 1],
                   c=colors[label], label=label_names[label],
                   alpha=0.6, s=15, edgecolors="none")
    ax.set_title(title, fontweight="bold", color="black")
    ax.legend()
    ax.set_xlabel("t-SNE dim 1")
    ax.set_ylabel("t-SNE dim 2")

fig_tsne.suptitle("t-SNE: CLS Embeddings — CE vs SCL", fontsize=13, fontweight="bold")
fig_tsne.tight_layout()
plt.show()

fig_tsne.savefig(os.path.join(output_dir_scl, "tsne_comparison.png"), dpi=150, bbox_inches="tight")
print("t-SNE comparison saved.")

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────
with open(metrics_path_ce,  "r", encoding="utf-8") as f:
    saved_ce  = json.load(f)
with open(metrics_path_scl, "r", encoding="utf-8") as f:
    saved_scl = json.load(f)

print(f"\n{'='*60}")
print(f"  FINAL SUMMARY")
print(f"{'='*60}")
print(f"  {'Metric':<25} {'RoBERTa-CE':>12} {'RoBERTa-SCL':>13} {'Δ':>10}")
print(f"  {'-'*62}")
for metric, key in [("Val Accuracy", "accuracy"), ("Val F1 (macro)", "f1_macro"),
                    ("Val Precision", "precision_macro"), ("Val Recall", "recall_macro")]:
    v_ce  = saved_ce["validation_results"][key]
    v_scl = saved_scl["validation_results"][key]
    print(f"  {metric:<25} {v_ce:>12.4f} {v_scl:>13.4f} {v_scl - v_ce:>+10.4f}")
print(f"  {'Train time (s)':<25} {saved_ce['training']['training_time_seconds']:>12.1f} {saved_scl['training']['training_time_seconds']:>13.1f}")
print(f"  {'Model size (MB)':<25} {saved_ce['model_size']['MB']:>12.4f} {saved_scl['model_size']['MB']:>13.4f}")

print(f"\n  Results saved in:")
print(f"    CE:  {output_dir_ce}")
print(f"    SCL: {output_dir_scl}")

print(f"\n  Files in SCL dir:")
for fname in sorted(os.listdir(output_dir_scl)):
    fsize = os.path.getsize(os.path.join(output_dir_scl, fname))
    print(f"    {fname} ({fsize:,} bytes)")